#MIXX​ – Food Demand Prediction & Waste Intelligence

###Validate the dataset

In [ ]:
import pandas as pd

# 1) Load the source CSV file
df = pd.read_csv("/mixx_synthetic_restaurant_data.csv")

# 2) Parse date and sort (important for time-based forecasting)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["dish_name", "date"]).reset_index(drop=True)

# 3) Quick inspection
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
display(df.head(10))

# 4) Data quality checks (must be OK before modeling)
print("\nMissing values per column:")
print(df.isna().sum())

print("\nDate range:", df["date"].min(), "to", df["date"].max())
print("Unique dishes:", df["dish_name"].nunique())
print("Unique categories:", df["category"].nunique())

# 5) Define the prediction target (what we will forecast next day)
TARGET = "sold_qty"
print("\nTarget column:", TARGET)
print("Target stats:")
display(df[TARGET].describe())

##Model dataset shape

In [ ]:
import numpy as np

# Make sure sorted correctly (very important)
df = df.sort_values(["dish_name", "date"]).reset_index(drop=True)

# Create next-day target per dish
df["target_next_day"] = df.groupby("dish_name")["sold_qty"].shift(-1)

#Create lag features (what model can use to predict tomorrow)

# Yesterday demand
df["lag_1"] = df.groupby("dish_name")["sold_qty"].shift(1)

# Demand 7 days ago (weekly pattern)
df["lag_7"] = df.groupby("dish_name")["sold_qty"].shift(7)

# 7-day rolling average (trend)
df["rolling_mean_7"] = (
    df.groupby("dish_name")["sold_qty"]
    .shift(1)
    .rolling(7)
    .mean()
)

#Remove rows where lag/target is NaN
df_model = df.dropna().reset_index(drop=True)

print("Model dataset shape:", df_model.shape)
display(df_model.head(10))

###Time-Based Train/Test Split

In [ ]:
split_date = "2025-11-01"

train = df_model[df_model["date"] < split_date]
test  = df_model[df_model["date"] >= split_date]

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain date range:", train["date"].min(), "to", train["date"].max())
print("Test date range:", test["date"].min(), "to", test["date"].max())

We will:

Train → January to October

Test → November & December

We used a time-based split to simulate real-world forecasting, where the model is trained on historical data and evaluated on unseen future data.

###Define Features & Target

In [ ]:
# Target
y_train = train["target_next_day"]
y_test  = test["target_next_day"]

# Feature columns
feature_cols = [
    "lag_1",
    "lag_7",
    "rolling_mean_7",
    "temperature_c",
    "is_holiday"
]

X_train = train[feature_cols]
X_test  = test[feature_cols]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

###Train & Compare Multiple Models

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    return {
        "Model": name,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "MAPE (%)": round(mape, 2)
    }

results = []

# Naive Baseline
# Predict tomorrow = today (lag_1)
baseline_pred = X_test["lag_1"]
results.append(evaluate_model("Naive Baseline", y_test, baseline_pred))

# Linear Regression

lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
results.append(evaluate_model("Linear Regression", y_test, lr_pred))


# Random Forest

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
results.append(evaluate_model("Random Forest", y_test, rf_pred))


# Show comparison table

results_df = pd.DataFrame(results)
display(results_df.sort_values("RMSE"))

We compared a naive baseline, a linear model, and a nonlinear ensemble model.
Linear Regression achieved the lowest Root Mean Squared Error (RMSE), indicating that demand patterns in our synthetic restaurant data are largely linear and driven by historical lag features. Therefore, we selected Linear Regression as the final production model.

We predict next-day sold_qty.

RMSE measures the typical prediction error in the same unit as sold_qty (here: “portions/units sold”).

Lower RMSE = better model.

RMSE is useful because a few very wrong predictions matter a lot in restaurant planning (they cause waste or shortage), and RMSE highlights that.

###Daily Retrain + Predict Next Day

In [ ]:
from sklearn.linear_model import LinearRegression
import pandas as pd

def mixx_daily_prediction_pipeline(df):

    df = df.sort_values(["dish_name", "date"]).reset_index(drop=True)

    # Create next-day target
    df["target_next_day"] = df.groupby("dish_name")["sold_qty"].shift(-1)

    # Lag features
    df["lag_1"] = df.groupby("dish_name")["sold_qty"].shift(1)
    df["lag_7"] = df.groupby("dish_name")["sold_qty"].shift(7)

    df["rolling_mean_7"] = (
        df.groupby("dish_name")["sold_qty"]
        .shift(1)
        .rolling(7)
        .mean()
    )

    df_model = df.dropna().reset_index(drop=True)

    feature_cols = [
        "lag_1",
        "lag_7",
        "rolling_mean_7",
        "temperature_c",
        "is_holiday"
    ]

    X = df_model[feature_cols]
    y = df_model["target_next_day"]

    # Train on all except last row
    model = LinearRegression()
    model.fit(X[:-1], y[:-1])

    # Use last row properly as DataFrame
    X_next = df_model.iloc[[-1]][feature_cols]

    next_day_prediction = model.predict(X_next)[0]

    return round(next_day_prediction, 2)

In [ ]:
next_prediction = mixx_daily_prediction_pipeline(df)
print("Predicted Next Day Demand:", next_prediction)

We implemented a rolling retraining system where the model updates automatically when new daily data is added.

###Predict Next Day for All Dishes

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd

def predict_next_day_all_dishes_smart(df):

    df = df.sort_values(["dish_name", "date"]).reset_index(drop=True)

    predictions = []

    for dish in df["dish_name"].unique():

        df_dish = df[df["dish_name"] == dish].copy()

        # If not enough history → fallback
        if len(df_dish) < 14:
            fallback_prediction = df_dish["sold_qty"].mean()
            predictions.append({
                "Dish Name": dish,
                "Predicted Next Day Demand": round(fallback_prediction, 2),
                "Model Used": "Fallback Average"
            })
            continue

        # Create features
        df_dish["target_next_day"] = df_dish["sold_qty"].shift(-1)
        df_dish["lag_1"] = df_dish["sold_qty"].shift(1)
        df_dish["lag_7"] = df_dish["sold_qty"].shift(7)

        df_dish["rolling_mean_7"] = (
            df_dish["sold_qty"]
            .shift(1)
            .rolling(7)
            .mean()
        )

        df_model = df_dish.dropna().reset_index(drop=True)

        feature_cols = [
            "lag_1",
            "lag_7",
            "rolling_mean_7",
            "temperature_c",
            "is_holiday"
        ]

        X = df_model[feature_cols]
        y = df_model["target_next_day"]

        model = LinearRegression()
        model.fit(X[:-1], y[:-1])

        X_next = df_model.iloc[[-1]][feature_cols]
        next_day_prediction = model.predict(X_next)[0]

        predictions.append({
            "Dish Name": dish,
            "Predicted Next Day Demand": round(next_day_prediction, 2),
            "Model Used": "Linear Regression"
        })

    return pd.DataFrame(predictions)

In [ ]:
dashboard_df = predict_next_day_all_dishes_smart(df)
display(dashboard_df.sort_values("Predicted Next Day Demand", ascending=False))

We trained an independent forecasting model per dish, allowing dish-level demand planning rather than global demand estimation.

The model predicts next-day demand in terms of number of units sold (servings), based on historical sales patterns.

###Holiday Intelligence

In [ ]:
import holidays

# Get years from dataset
data_years = sorted(df["date"].dt.year.unique())

# Also include next year for forecasting (important for demo!)
next_year = max(data_years) + 1

all_years = data_years + [next_year]

# Create Finland holiday calendar for all required years
fi_holidays = holidays.Finland(years=all_years)

print("Holiday years covered:", all_years)

###Weather & Holiday actually affect tomorrow’s prediction (forecast-aware)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

def predict_next_day_all_dishes_with_forecast(df, temp_tomorrow_c: float, date_tomorrow: str):
    """
    Predict next-day demand per dish using:
      - lag features from history
      - TOMORROW's temperature + Finland holiday flag (auto)
    """

    date_tomorrow = pd.to_datetime(date_tomorrow)

    # Finland holiday auto flag for tomorrow (uses the fi_holidays object from Step 9)
    is_holiday_tomorrow = 1 if date_tomorrow in fi_holidays else 0

    df = df.sort_values(["dish_name", "date"]).reset_index(drop=True)

    predictions = []

    for dish in df["dish_name"].unique():
        df_dish = df[df["dish_name"] == dish].copy()

        # Fallback if new dish / too little history
        if len(df_dish) < 14:
            predictions.append({
                "Dish Name": dish,
                "Predicted Next Day Demand": round(df_dish["sold_qty"].mean(), 2),
                "Model Used": "Fallback Average",
                "Temp Tomorrow (C)": temp_tomorrow_c,
                "Holiday Tomorrow": is_holiday_tomorrow
            })
            continue

        # Features + target
        df_dish["target_next_day"] = df_dish["sold_qty"].shift(-1)
        df_dish["lag_1"] = df_dish["sold_qty"].shift(1)
        df_dish["lag_7"] = df_dish["sold_qty"].shift(7)
        df_dish["rolling_mean_7"] = df_dish["sold_qty"].shift(1).rolling(7).mean()

        df_model = df_dish.dropna().reset_index(drop=True)

        feature_cols = ["lag_1", "lag_7", "rolling_mean_7", "temperature_c", "is_holiday"]

        X = df_model[feature_cols]
        y = df_model["target_next_day"]

        model = LinearRegression()
        model.fit(X[:-1], y[:-1])

        # Build tomorrow feature row from last known lags, but override weather/holiday with TOMORROW values
        X_next = df_model.iloc[[-1]][feature_cols].copy()
        X_next["temperature_c"] = temp_tomorrow_c
        X_next["is_holiday"] = is_holiday_tomorrow

        pred = model.predict(X_next)[0]

        predictions.append({
            "Dish Name": dish,
            "Predicted Next Day Demand": round(pred, 2),
            "Model Used": "Linear Regression",
            "Temp Tomorrow (C)": temp_tomorrow_c,
            "Holiday Tomorrow": is_holiday_tomorrow
        })

    return pd.DataFrame(predictions).sort_values("Predicted Next Day Demand", ascending=False)

In [ ]:
tomorrow_date = "2026-01-01"
tomorrow_temp_c = 2.0

dashboard_forecast = predict_next_day_all_dishes_with_forecast(df, tomorrow_temp_c, tomorrow_date)
display(dashboard_forecast)

1️⃣ Problem Definition: The goal is to predict next-day dish-level demand to reduce food waste and improve kitchen planning.

2️⃣ Model Design: We implemented a rolling retraining system.
Each dish has its own forecasting model trained using lag features, temperature, and holiday effects.

keywords:

Time-series forecasting

Lag-based features

External regressors (weather + holiday)

3️⃣ Why Weather Matters: Weather influences customer behavior.
For example, colder days may increase demand for warm dishes like soups or rice-based meals.

4️⃣ Why Holiday Matters

Holiday effect is automatically detected using Finland’s national holiday calendar.
On holidays, restaurant demand patterns differ significantly.

system:

Automatically checks if tomorrow is a Finland holiday

Injects that information into the model

That is very strong design.

5️⃣ Interpretation of This Specific Output

Rice → 147
Chicken Curry → 130
Beef Stew → 109

explain:

Because tomorrow is a holiday and temperature is low (2°C), the model predicts higher demand compared to non-holiday average days.

can even compare:

Run with temperature = 15°C

Run with holiday = 0

And show how numbers change.

**“Staff inputs today’s cooked & sold quantity → system updates → retrains → predicts tomorrow again.”**

###Automatic Weather Fetch

In [ ]:
import requests

def get_tomorrow_temperature(lat=60.1699, lon=24.9384):
    """
    Default coordinates = Helsinki, Finland.
    You can change if needed.
    """

    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}"
        f"&daily=temperature_2m_max"
        f"&timezone=Europe/Helsinki"
    )

    response = requests.get(url)
    data = response.json()

    tomorrow_temp = data["daily"]["temperature_2m_max"][1]  # index 1 = tomorrow

    return tomorrow_temp

###Build Staff Daily Input Function

In [ ]:
def add_daily_data_and_predict(df, date_today, temp_today_c, sold_dict, cooked_dict,
                               new_dish_meta=None):
    """
    new_dish_meta example:
    {
      "Salmon Soup": {"category":"Main", "price_level":"High"},
      "Fruit Salad": {"category":"Dessert", "price_level":"Low"}
    }
    """

    date_today = pd.to_datetime(date_today)
    new_rows = []

    if new_dish_meta is None:
        new_dish_meta = {}

    for dish in sold_dict.keys():
        sold = sold_dict[dish]
        cooked = cooked_dict[dish]
        wasted = cooked - sold

        # If dish exists in historical data -> take category/price_level from there
        if dish in df["dish_name"].values:
            meta_row = df[df["dish_name"] == dish].iloc[0]
            category = meta_row["category"]
            price_level = meta_row["price_level"]
        else:
            # New dish -> take metadata from new_dish_meta, else default
            category = new_dish_meta.get(dish, {}).get("category", "Main")
            price_level = new_dish_meta.get(dish, {}).get("price_level", "Medium")

        new_rows.append({
            "date": date_today,
            "weekday": date_today.day_name(),
            "dish_name": dish,
            "category": category,
            "price_level": price_level,
            "temperature_c": temp_today_c,
            "is_holiday": 1 if date_today in fi_holidays else 0,
            "cooked_qty": cooked,
            "sold_qty": sold,
            "wasted_qty": wasted
        })

    df_updated = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

    # Predict tomorrow (forecast-aware)
    tomorrow_date = (date_today + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    tomorrow_temp = get_tomorrow_temperature()

    dashboard = predict_next_day_all_dishes_smart(df_updated)  # smart handles new dishes safely

    return df_updated, dashboard, tomorrow_date, tomorrow_temp

###DEMO 1: BASELINE FORECAST (Historical Data Only)

In [ ]:
forecast_temp = get_tomorrow_temperature()

historical_dashboard = predict_next_day_all_dishes_smart(df)

print("📊 Forecast Based ONLY on Historical Data")
print("🌡 Forecast temperature:", forecast_temp, "°C")

display(historical_dashboard.sort_values("Predicted Next Day Demand", ascending=False))

Staff enters today’s actual sales.
Cooked quantity was based on yesterday’s prediction.
The system automatically detects weather forecast.
It retrains the model with new data.
Then it generates updated demand forecast for tomorrow.

###DEMO 2: Staff Daily Input with new dish

In [ ]:
today_date = "2026-01-01"
today_temp = get_tomorrow_temperature()

sold_today = {
    "Rice": 150,
    "Chicken Curry": 128,
    "Beef Stew": 105,
    "Veg Pasta": 95,
    "Chocolate Pudding": 70,
    "Salmon Soup": 40           # ✅ NEW DISH
}

cooked_today = {
    "Rice": 147,
    "Chicken Curry": 130,
    "Beef Stew": 109,
    "Veg Pasta": 98,
    "Chocolate Pudding": 75,
    "Salmon Soup": 45           # cooked a bit more than sold
}

new_dish_meta = {
    "Salmon Soup": {"category": "Main", "price_level": "High"}
}

df, dashboard, tomorrow_date, tomorrow_temp = add_daily_data_and_predict(
    df, today_date, today_temp, sold_today, cooked_today, new_dish_meta
)

print("Tomorrow date:", tomorrow_date, "| Forecast temp:", tomorrow_temp)
display(dashboard.sort_values("Predicted Next Day Demand", ascending=False))

**(New dish) = staff inputs sold/cooked, and a new dish appears with Model Used = Fallback Average (because there’s no history for that dish yet, so your system uses a safe default).**

###DEMO 3: Staff Daily Input with new additional dish

In [ ]:
"""
today_date = "2026-03-12"
today_temp = get_tomorrow_temperature()

# Staff daily input
sold_today = {
    "Rice": 155
    "Chicken Curry": 120,
    "Beef Stew": 108,
    "Veg Pasta": 93,
    "Chocolate Pudding": 72,
    "Salmon Soup": 38
}

cooked_today = {
    "Rice": 150,
    "Chicken Curry": 125,
    "Beef Stew": 110,
    "Veg Pasta": 96,
    "Chocolate Pudding": 75,
    "Salmon Soup": 42
}

# NEW ADDITIONAL DISH
new_dish_meta = {
    "Shrimp Pasta": {"category": "Main", "price_level": "High"}
}

# Run smart update + retrain
df, dashboard, tomorrow_date, tomorrow_temp = add_daily_data_and_predict(
    df,
    today_date,
    today_temp,
    sold_today,
    cooked_today,
    new_dish_meta
)

print("Tomorrow date:", tomorrow_date, "| Forecast temp:", tomorrow_temp)

display(
    dashboard.sort_values("Predicted Next Day Demand", ascending=False)
)"""